# Machine Learning as an Aid Against Phishing and Fraud
---
## Authors:
Alan Becker,
Alexander Zambie,
Brennan Slate,
and Lazaro Martinez Jimenez


# Introduction

A phishing email is a fraudulent message from a cybercriminal who impersonates a trusted source to trick you into revealing sensitive information, such as passwords, credit card numbers, or bank details. In this report predictions will be made to distiguish if an email is phishing/fraudulent, or a real email.


## Dataset Choice
This project is using the [Phishing Email Dataset](https://www.kaggle.com/datasets/naserabdullahalam/phishing-email-dataset) from Kaggle. This dataset contains approximately 82,500 emails, with 42,891 labeled as phishing and 39,595 as legitimate. The emails are sourced from Enron, Ling, CEAS, Nazario, SpamAssassin, and Nigerian Fraud emails. For this project, we will focus on the CEAS subset, as it provides more relevant features and is less dated.


## Prediction Goal and Features
The goal of this project is to compare different machine learning models that can distinguish between phishing and legitimate emails based on their content.


We plan to use the columns `subject`, `body`, `sender`, and `urls` to predict the `label` column, which indicates whether an email is phishing or legitimate.









In [ ]:
# packages
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.metrics import cohen_kappa_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
import xgboost as xgb

## Read Data

In [ ]:
df = pd.read_csv('data/CEAS_08.csv')

# Data Exploration and Cleaning

Looking at the data, there are 2 columns with missing values. 462 instances in the "receiver" column, and 28 in the "subject" column. Majority of the types are objects, with labesl and url's being int64.



In [ ]:
na_per_col = df.isna().sum().sort_values(ascending=False)
print(na_per_col)
print(df.info())

## Data Preprocessing


Preparing the data for modeling by dropping unnecessary columns and handling missing values. Receiver and date will be dropped, as they do not have much impact on determining if an email is phishing or legitimate. Also filled the NA values of subject with ''. Also the Object types of sender, receiver, subject, and url's were changed to strings for future use. 

In [ ]:
df = df.drop(["receiver", "date"], axis=1)
df = df.fillna('')
df["urls"] = df["urls"].astype("string").fillna("").str.strip()
df["sender"] = df["sender"].astype("string").fillna("").str.strip()
df["subject"] = df["subject"].astype("string").fillna("").str.strip()


## Data Exploration and Visualization


In [ ]:
df.info()
df.head()

# Count Plot of Features

Although there are imbalances in the distribution of some of these features, they capture meaningful behavioral differences between legitimate and phishing emails, so these features are kept for modeling.

In [ ]:
url_pattern = r"(https?://|www\.)\S+" 

df["n_urls"] = (
    df["urls"].str.count(url_pattern) +
    df["subject"].str.count(url_pattern) +
    df["body"].str.count(url_pattern)
)

df["subject_len"] = df["subject"].str.len()
df["body_len"] = df["body"].str.len()

# Legit vs Phishing Count
count = df['label'].value_counts()
bars = plt.bar(count.index, count.values, color=['darkred', 'darkblue'])
plt.title('Distribution of Legitimate vs. Phishing Emails')
plt.xlabel('Email Type')
plt.ylabel('Count')
plt.xticks([0, 1], ['Legitimate (0)', 'Phishing (1)'])
plt.bar_label(bars, fmt='%d', fontsize=10)
plt.show()

# Subject length
plt.figure()
df["subject_len"].plot(kind="hist", bins=50)
plt.title("Subject Length Distribution")
plt.xlabel("characters")
plt.ylabel("count")
plt.show()


#Body length
plt.figure()
df["body_len"].clip(upper=df["body_len"].quantile(0.99)).plot(kind="hist", bins=50)
plt.title("Body Length Distribution")
plt.xlabel("characters")
plt.ylabel("count")
plt.show()

# URL's per email count
plt.figure()
df["n_urls"].clip(upper=10).plot(kind="hist", bins=30)
plt.title("URL Count")
plt.xlabel("# of URLs")
plt.ylabel("count")
plt.show()



In [ ]:
plot_data = pd.crosstab(df['label'], df['urls'])
fig, ax = plt.subplots(figsize=(10, 6))
plot_data.plot(kind='bar', ax=ax, color=['darkblue', 'darkred'])
ax.set_title('URL Presence vs. Email Legitimacy', fontsize=16)
ax.set_xlabel('Email Type', fontsize=14)
ax.set_ylabel('Count', fontsize=14)
ax.set_xticklabels(['Legitimate (0)', 'Phishing (1)'])
ax.legend(title='URLs Present', labels=['No (0)', 'Yes (1)'], loc='upper right')
for container in ax.containers:
    ax.bar_label(container, fmt='%d', fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

## Machine Learning


In [ ]:
df['predictor'] = (df['subject'] + ' ' + df['body'] + ' ' + df['sender']) + ' ' + df['urls'].astype(str)
vectorizer = TfidfVectorizer()
X = df['predictor']
y = df['label'].values
# Splitting the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Applying TFIDF vectorizer after splitting the data
X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

In [ ]:
# Function to evaluate models
def eval_model(model, X_train, y_train, X_test, y_test):
    start_time = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start_time

    pred_time_start = time.time()
    y_pred = model.predict(X_test)
    pred_time = time.time() - pred_time_start
    y_prob = model.predict_proba(X_test)[:, 1]

    model_names = {
      "MultinomialNB": "Naive Bayes",
      "LogisticRegression": "Logistic Regression",
      "KNeighborsClassifier": "K-Nearest Neighbors"
    }

    train_score = model.score(X_train, y_train)
    test_score = model.score(X_test, y_test)

    # Confusion Matrix
    labels = sorted(np.unique(y_test))
    cm = confusion_matrix(y_test, y_pred, labels=labels)
    ConfusionMatrixDisplay(cm, display_labels=labels).plot(values_format="d", cmap="Blues")
    plt.title(f"Confusion Matrix for {model_names[model.__class__.__name__]}")
    plt.show()

    # ROC curve
    pos_label = labels[-1]
    y_bin = (y_test == pos_label).astype(int)
    fpr, tpr, _ = roc_curve(y_bin, y_prob)
    auc_nb = roc_auc_score(y_bin, y_prob)

    plt.figure()
    plt.plot(fpr, tpr, label=f"{model_names[model.__class__.__name__]} (AUC = {auc_nb:.2f})")
    plt.plot([0, 1], [0, 1], "--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve for {model_names[model.__class__.__name__]}")
    plt.legend(loc="lower right")
    plt.show()

    # Model performance metrics
    print(f"Model Training Time: {train_time:.4f} seconds")
    print(f"Prediction Time: {pred_time:.4f} seconds")
    print("Train Accuracy: {:.2f}%".format(train_score*100))
    print("Test Accuracy: {:.2f}%".format(test_score*100))
    print("Cohen's Kappa:", round(cohen_kappa_score(y_test, y_pred), 3))
    print(model_names[model.__class__.__name__] + " Classification Report")
    print(classification_report(y_test, y_pred, zero_division=0))


## KNN (K-Nearest Neighbors) Approach

In [ ]:
knn_model = KNeighborsClassifier()
eval_model(knn_model, X_train, y_train, X_test, y_test)

## Naives Bayes Approach

In [ ]:
naivebayes_model = MultinomialNB()
eval_model(naivebayes_model, X_train, y_train, X_test, y_test)

## Logistic Regression Approach


In [ ]:
model = LogisticRegression()
eval_model(model, X_train, y_train, X_test, y_test)

# Confusion Matrix


The confusion matrix shows how our Logistic Regression model tells the difference between legitimate and phishing emails. Out of all legitimate emails, only 29 were misclassified as phishing, while 971 were correctly identified as legitimate. And then out of all phishing emails, 982 were correctly detected as phishing, and only 18 were mistakenly labeled as legitimate.

These results show that the model works really well in both detexting phishing and legitimate emails. 

# Random Forest Model

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    max_depth=None,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]  # for ROC curve

# matrix
labels = sorted(np.unique(y_test))
cm = confusion_matrix(y_test, y_pred_rf, labels=labels)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
fig, ax = plt.subplots()
disp.plot(ax=ax, values_format="d", cmap="Blues")
ax.set_title("Confusion Matrix — Random Forest")
plt.show()

print("Classification report for Random Forest")
print(classification_report(y_test, y_pred_rf, zero_division=0))

# roc
fpr, tpr, _ = roc_curve(y_test, y_prob_rf)
roc_auc = roc_auc_score(y_test, y_prob_rf)

plt.figure()
plt.plot(fpr, tpr, label='Random Forest (AUC = {:.2f})'.format(roc_auc), color='forestgreen')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Random Forest')
plt.legend(loc="lower right")
plt.show()

# Linear SVC Approach

In [ ]:
# Initialize, Fit, & Evaluate
svc_model = LinearSVC(random_state=42)
eval_model(svc_model, X_train, y_train, X_test, y_test)

# Get predictions
y_pred_svc = svc_model.predict(X_test)
print("Kappa Coefficient:", round(cohen_kappa_score(y_test, y_pred_svc), 3))

# LinearSVC uses decision_function, not predict_proba
y_decision_svc = svc_model.decision_function(X_test)

# Display Confusion Matrix for LinearSVC
labels = sorted(np.unique(y_test))
cm_svc = confusion_matrix(y_test, y_pred_svc, labels=labels)
ConfusionMatrixDisplay(cm_svc, display_labels=labels).plot(values_format="d", cmap="Oranges")
plt.title("Confusion Matrix for Linear SVC")
plt.show()

# Calculate ROC metrics using decision function
fpr_svc, tpr_svc, _ = roc_curve(y_test, y_decision_svc)
auc_svc = roc_auc_score(y_test, y_decision_svc)

# Plot ROC Curve
plt.figure()
plt.plot(fpr_svc, tpr_svc, label=f"Linear SVC (AUC = {auc_svc:.2f})")
plt.plot([0, 1], [0, 1], "--") # Diagonal line
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve for Linear SVC")
plt.legend(loc="lower right")
plt.show()

# XGBoost Approach

In [ ]:
# Initialize, Fit & Evaluate
xgb_model = xgb.XGBClassifier(random_state=42, eval_metric='logloss')
eval_model(xgb_model, X_train, y_train, X_test, y_test)

# Get predictions
y_pred_xgb = xgb_model.predict(X_test)
print("Kappa Coefficient:", round(cohen_kappa_score(y_test, y_pred_xgb), 3))

# Get probabilities for ROC curve
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

# Display Confusion Matrix for XGBoost
labels = sorted(np.unique(y_test))
cm_xgb = confusion_matrix(y_test, y_pred_xgb, labels=labels)
ConfusionMatrixDisplay(cm_xgb, display_labels=labels).plot(values_format="d", cmap="Purples")
plt.title("Confusion Matrix for XGBoost")
plt.show()

# Calculate ROC metrics using predict_proba
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_prob_xgb)
auc_xgb = roc_auc_score(y_test, y_prob_xgb)

# Plot ROC Curve
plt.figure()
plt.plot(fpr_xgb, tpr_xgb, label=f"XGBoost (AUC = {auc_xgb:.2f},)", color='purple')
plt.plot([0, 1], [0, 1], "--", color='black') # Diagonal line
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve for XGBoost")
plt.legend(loc="lower right")
plt.show()